In [1]:
# -q(--quiet): 설치 중 출력되는 긴 로그를 최소화하여 셀 출력을 깔끔하게
# -U(--upgrade): 이미 colab 환경에 설치되어 있는 패키지라도 최신 버전으로 강제 업그레이드
# transformers: Hugging Face의 핵심 라이브러리
# accelerate: Pytorch 분산/하드웨어 가속 유틸리티
# qwen-vl-utils[decord]: Qwen-VL 계열 모델 데이터 전처리 라이브러리
# datasets:  Hugging Face의 대용량 데이터 로더
# huggingface_hub: Hugging Face 플랫폼 연동 클라이언트
# pillow: 파이썬 표준 이미지 처리 라이브러리(PIL)
!pip install -q -U transformers accelerate qwen-vl-utils[decord] datasets huggingface_hub # pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 77.5 MB/s eta 0:00:00


In [2]:
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
MODEL_REVISION = "ebb281ec70b05090aa6165b016eac8ec08e71b17"

DATASET_ID = "MMMU/MMMU"
DATASET_REVISION = "98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68"

SUBJECTS = [
    "Accounting", "Agriculture", "Architecture_and_Engineering", "Art", "Art_Theory",
    "Basic_Medical_Science", "Biology", "Chemistry", "Clinical_Medicine", "Computer_Science",
    "Design", "Diagnostics_and_Laboratory_Medicine", "Economics", "Electronics",
    "Energy_and_Power", "Finance", "Geography", "History", "Literature", "Manage",
    "Marketing", "Materials", "Math", "Mechanical_Engineering", "Music", "Pharmacy",
    "Physics", "Psychology", "Public_Health", "Sociology",
]

In [3]:
import torch

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

MIN_PIXELS = 256 * 28 * 28   # 이미지당 최소 token 수 하한
MAX_PIXELS = 1024 * 28 * 28  # 이미지당 최대 token 수 상한

# AutoModelForImageTextToText: 이미지와 질문을 입력받아 답변 텍스트를 만들어내는 멀티모달 AI 모델 본체
# AutoProcessor: 이미지와 텍스트를 AI 모델이 이해할 수 있는 숫자 형태로 자동 변환(전처리)해 주는 도구
from transformers import AutoModelForImageTextToText, AutoProcessor

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    torch_dtype="auto",       # 모델 제작자가 리포지토리에 저장해 둔 원래의 데이터 타입을 그대로 가져와 용량 최적화
    device_map="auto"        # 사용 가능한 GPU와 CPU의 용량을 계산하여 모델을 가장 최적의 하드웨어 위치에 자동으로 분할 배치
    )
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    min_pixels = MIN_PIXELS,
    max_pixels = MAX_PIXELS
    )

# do_sample: true -> 확률적 샘플링을 활성화
# repetition_penalty: 1.0 -> 동일한 단어나 문장이 반복되는 현상을 억제하는 패널티 계수. 1.0은 패널티를 전혀 주지 않는 기본 상태
# temperature: 모델이 예측한 다음 토큰들의 확률 분포를 부드럽게 펴거나 뾰족하게 만드는 온도 파라미터. 0에 가까울수록 가장 확률 높은 정답 위주로 출력
# top_k: 20 -> 다음 단어를 고를 때 확률이 가장 높은 상위 20개 단어 후보만 남기고 나머지는 탈락시키는 필터링
# top_p: 0.8 -> 확률 상위 단어들을 누적해 더했을 때, 누적 확률의 합이 80%가 되는 지점까지만 후보군으로 유지하는 동적 필터링

# 작동 흐름 예시: top_k=20으로 상위 20개를 먼저 거른 뒤-> top_p=0.8로 누적 확률 80% 안의 단어들만 최종 후보로 추리고
# temperature=0.7로 조율된 확률에 다라 do_sample=True로 하나를 뽑아냄
print(model.generation_config) # AI 모델이 답변을 생성할 때 사용하는 하이퍼파라미터 설정값들을 출력

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.0,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}



In [4]:
print(getattr(model.generation_config, "presence_penalty", None))
# None -> generation_config.json 파일에 presence_penalty 키 자체가 정의되어 있지 않음

None


In [5]:
# load_dataset: Hugging Face Hub에서 지정된 데이터셋의 특정 서브셋과 분할 데이터를 다운로드하고 캐싱하는 함수
# concatenate_datasets: 분할된 여러 개의 Dataset 객체를 하나의 큰 Dataset으로 물리적으로 이어 붙여 병합할 때 사용하는 함수
from datasets import load_dataset, concatenate_datasets

all_splits = []
for subj in SUBJECTS:
    ds = load_dataset(DATASET_ID, subj, split="validation", revision=DATASET_REVISION)
    ds = ds.add_column("subject", [subj] * len(ds))
    all_splits.append(ds)
    print(subj, len(ds))

assert sum(len(d) for d in all_splits) == 900, "과목당 30문제, 총 900문제여야 함"

README.md:   0%|          | 0.00/42.7k [00:00<?, ?B/s]

Accounting/dev-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  273kB            

Accounting/dev-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Accounting/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.54MB            

Accounting/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

...

Sociology/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/30 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/252 [00:00<?, ? examples/s]

Sociology 30


In [6]:
import re # 정규표현식(Regular Expression) 모듈
import string # 파이썬 표준 문자열 상수 모듈
from PIL import Image # 이미지를 열고, 수정하고, 저장할 때 사용하는 이미지 처리 라이브러리

def build_prompt_and_images(example):
    question = example["question"]
    q_type = example["question_type"] # "multiple-choice" or "open"

    # 이미지 <image 1>, <image 2> ... 순서대로 수집
    images = []
    for i in range(1, 8):
        img = example.get(f"image_{i}")
        if img is not None:
            images.append(img.convert("RGB") if isinstance(img, Image.Image) else img)

    if q_type == "multiple-choice":
        raw_options = example["options"] # 데이터 샘플의 객관식 선택지 데이터를 가져옴
        if isinstance(raw_options, str):
            raw_options = eval(raw_options) # MMMU 데이터셋 특성상 str(list) 형태
        letters = list(string.ascii_uppercase[: len(raw_options)]) # 선택지 개수만큼 대문자 알파벳을 슬라이싱하여 리스트로 만듦
        options_block = "\n".join(f"{l}. {opt}" for l, opt in zip(letters, raw_options))
        prompt_text = (
            f"Question: {question}\n"
            f"Options:\n{options_block}\n"
            f"Respond with ONLY the single letter of the correct option (e.g. 'A'). "
            f"Do not provide any explanation or reasoning."
        )
    else: # open
        letters = []
        prompt_text = (
            f"Question: {question}\n"
            f"Answer the question directly with a short, precise answer "
            f"(a number, word, or short phrase). Do not provide any explanation or reasoning."
        )

    # prompt_text: 모델에 입력할 최종 조합 텍스트
    # images: 유효한 RGB 이미지 객체들이 담긴 리스트
    # letters: multiple-choice면 알파벳 목록, open이면 빈 리스트
    # q_type: 채점 분기용
    return prompt_text, images, letters, q_type

In [7]:
# GEN_KWARDS 출처: https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct
GEN_KWARGS = dict(
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    repetition_penalty=1.0,
#    presence_penalty=1.5,
)
def run_one(example):
    prompt_text, images, letters, qtype = build_prompt_and_images(example)
    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": prompt_text})
    messages = [{"role": "user", "content": content}]
    # apply_chat_template: 텍스트(그리고 이미지)를 컴퓨터(모델)가 이해할 수 있는 숫자로 변환해줌
    inputs = processor.apply_chat_template(
        # tokenize=True: 텍스트를 토큰화하여 고유 정수ID로 바꿈
        # add_generation_prompt: 모델이 답변을 시작할 수 있도록 대화 템플릿의 끝에 어시스턴트 시작 태그를 자동으로 덧붙임
        # return_dict=True: 결과물을 딕셔너리 형태로 반환
        # return_tensors="pt": 변환된 숫자 데이터를 파이토치의 텐서 형식으로 출력
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)

    with torch.no_grad(): # 추론 단계이므로 역전파를 위한 계산 그래프 및 기울기 추적을 비활성화
        out_ids = model.generate(**inputs, **GEN_KWARGS) # 입력 텐서와 생성 파라미터를 풀어 넣어 autoregressive하게 다음 토큰들을 최대 32개까지 생성
    gen_ids = out_ids[:, inputs["input_ids"].shape[1]:] # inputs["input_ids"].shape[1]은 입력값(질문 프롬프트)의 총 토큰 길이
    # 토큰 ID를 다시 사람이 읽을 수 있는 문자열로 복원
    # skip_special_tokens=True: 내부 특수 제어 토큰들을 화면에 표시하지 않고 제거
    text = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    return text, letters, qtype # 모델이 생성한 최종 답변 텍스트와 정답 검증용 선택지 리스트를 반환

In [8]:
def parse_answer(generated_text, letters, q_type):
    text = generated_text.strip()

    if q_type == "multiple-choice":
      # 1) "A" 단독 또는 "A." "A)" 처럼 맨 앞에 오는 패턴
      m = re.match(r"^\(?([A-J])\)?[\.\:\)]?", text) # re.match: 문자열의 시작 부분부터 패턴이 일치하는지 검사
      if m and m.group(1) in letters: # m.group(1): A부터 J 사이의 대문자 1글자를 찾아냄
          return m.group(1)
      # 2) 텍스트 어디든 "answer is A" 류 패턴
      m = re.search(r"(?:answer is|answer:|option)\s*\(?([A-J])\)?", text, re.IGNORECASE) # re.search: 문자열의 시작뿐만 아니라 전체 텍스트 어디든 해당 패턴이 나타나는지 검색
      if m and m.group(1).upper() in letters:
          return m.group(1).upper()
      # 3) fallback: 텍스트에 등장하는 첫 letter 후보
      for ch in text:
          if ch in letters:
              return ch
      return None  # 파싱 실패 -> 오답 처리

    else: # open -> 정규화 후 gold와 문자열 비교 (main loop에서 gold도 동일 정규화 필요)
      return text.strip().lower().rstrip(".")

In [9]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/mmmu_baseline"
os.makedirs(SAVE_DIR, exist_ok=True)
log_path = f"{SAVE_DIR}/raw_logs.jsonl"

from collections import defaultdict
from tqdm.auto import tqdm
import json, time, gc

def normalize_open(ans: str) -> str:
    return str(ans).strip().lower().rstrip(".")

# 이미 처리된 id 불러오기 (이어서 진행)
done_ids = set()
results = defaultdict(lambda: {"correct": 0, "total": 0}) # 새로운 key가 처음 들어올 때 KeyError 없이 자동으로 딕셔너리 템플릿 생성
if os.path.exists(log_path):
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            done_ids.add(r["id"])
            results[r["subject"]]["total"] += 1
            results[r["subject"]]["correct"] += int(
                r["pred"] == r["gold"] if r["q_type"] == "multiple-choice"
                else (r["pred"] is not None and r["pred"] == normalize_open(r["gold"]))
            )
    print(f"이미 처리된 문제 수: {len(done_ids)} -> 이어서 진행")

start = time.time()
# 바깥쪽 루프에서는 30개 과목의 데이터셋을 하나씩 가져오고
# 안쪽 루프에서는 과목 안에 들어있는 30개의 문제를 가져옴
with open(log_path, "a") as f:
    for ds in all_splits:
        subj_name = ds[0]["subject"]
        for ex in tqdm(ds, desc=subj_name):
            if ex.get("id") in done_ids:   # 이미 완료한건 패스
                continue
            subj = ex["subject"]
            try:
                gen_text, letters, q_type = run_one(ex)
                pred = parse_answer(gen_text, letters, q_type)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                gc.collect() # 호출하는 즉시 불필요한 메모리(참조되지 않은 객체)를 강제로 해제
                gen_text, pred, letters, q_type = "[OOM_ERROR]", None, [], ex["question_type"]

            gold_raw = ex["answer"]
            if q_type == "multiple-choice":
                gold = gold_raw
                is_correct = (pred == gold)
            else: # open
                gold = normalize_open(gold_raw)
                is_correct = (pred is not None and pred == gold)

            results[subj]["total"] += 1
            results[subj]["correct"] += int(is_correct)

            row = {
                "id": ex.get("id"), "subject": subj, "q_type": q_type,
                "gold": gold_raw, "pred": pred, "raw_output": gen_text,
            }
            # json.dumps: 파이썬 딕셔너리를 "텍스트 문자열"로 변환해주는 함수
            # ensure_ascii=False: 한글이나 유니코드 특수문자가 원래 글자 그대로 저장되도록 함
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            f.flush()

        # 과목 끝날 때마다 메모리 정리
        torch.cuda.empty_cache()
        gc.collect()

elapsed = time.time() - start
print(f"Total time: {elapsed/60:.1f} min")

Mounted at /content/drive
이미 처리된 문제 수: 426 -> 이어서 진행


Accounting:   0%|          | 0/30 [00:00<?, ?it/s]

Agriculture:   0%|          | 0/30 [00:00<?, ?it/s]

Architecture_and_Engineering:   0%|          | 0/30 [00:00<?, ?it/s]

Art:   0%|          | 0/30 [00:00<?, ?it/s]

...

Physics:   0%|          | 0/30 [00:00<?, ?it/s]

Psychology:   0%|          | 0/30 [00:00<?, ?it/s]

Public_Health:   0%|          | 0/30 [00:00<?, ?it/s]

Sociology:   0%|          | 0/30 [00:00<?, ?it/s]

Total time: 125.0 min


In [10]:
import re, ast # ast(Abstract Syntax Tree): 프로그래밍 언어로 작성된 소스 코드의 구조를 컴퓨터가 이해하기 쉽게 트리 형태로 표현한 객체 모델
import pandas as pd

def gold_candidates(gold_raw):
    s = str(gold_raw).strip()
    if s.startswith("[") and s.endswith("]"):
        try:
            return [str(x) for x in ast.literal_eval(s)]
        except Exception:
            return [s]
    return [s]

def parse_number(s):
    s = str(s).strip()
    m = re.fullmatch(r"-?\d+\.?\d*\s*/\s*\d+\.?\d*\s*\w*", s) # 분수인지 확인
    if m: # m이 분수라면
        num_str = re.match(r"-?\d+\.?\d*", s).group()
        denom_str = re.search(r"/\s*(\d+\.?\d*)", s).group(1)
        try:
            return float(num_str) / float(denom_str)
        except Exception:
            return None
    nums = re.findall(r"-?\d+\.?\d*", s) # m이 분수가 아니라면
    if len(nums) == 1:
        try:
            return float(nums[0])
        except Exception:
            return None
    return None

def is_open_correct(pred, gold_raw, rel_tol=0.02, abs_tol=0.01): # 범위 내 오차 허용
    if pred is None:
        return False
    candidates = gold_candidates(gold_raw)
    pred_norm = pred.strip().lower().rstrip(".")
    for c in candidates:
        if pred_norm == c.strip().lower().rstrip("."):
            return True
    pred_num = parse_number(pred)
    for c in candidates:
        gold_num = parse_number(c)
        if pred_num is not None and gold_num is not None:
            if abs(pred_num - gold_num) <= max(abs_tol, abs(gold_num) * rel_tol):
                return True
    return False

with open(log_path) as f:
    all_rows = [json.loads(l) for l in f]

final_results = defaultdict(lambda: {"correct": 0, "total": 0})
for r in all_rows:
    subj = r["subject"]
    if r["q_type"] == "multiple-choice":
        correct = (r["pred"] == r["gold"])
    else:
        correct = is_open_correct(r["pred"], r["gold"])
    final_results[subj]["total"] += 1
    final_results[subj]["correct"] += int(correct)

rows_out = []
for subj in SUBJECTS:
    c, t = final_results[subj]["correct"], final_results[subj]["total"]
    rows_out.append({"Subject": subj, "Data Num": t, "Acc": round(c/t*100, 2)})

df = pd.DataFrame(rows_out)
overall = df["Acc"].mean()
print(df)
print(f"\nOverall (macro avg): {overall:.2f}")

                                Subject  Data Num    Acc
0                            Accounting        30  53.33
1                           Agriculture        30  50.00
2          Architecture_and_Engineering        30  46.67
3                                   Art        30  66.67
4                            Art_Theory        30  83.33
5                 Basic_Medical_Science        30  66.67
6                               Biology        30  46.67
7                             Chemistry        30  30.00
8                     Clinical_Medicine        30  60.00
9                      Computer_Science        30  53.33
10                               Design        30  86.67
11  Diagnostics_and_Laboratory_Medicine        30  33.33
12                            Economics        30  50.00
13                          Electronics        30  50.00
14                     Energy_and_Power        30  40.00
15                              Finance        30  30.00
16                            G